# Hardware Doctor Live Diagnostic

This notebook runs the new AWR2944 hardware doctor to verify your system setup and physical connections. It performs diagnostic checks without mutating any hardware state or executing captures.

In [1]:
from awr2944_dca.lab import RadarProject
from rich.console import Console

project = RadarProject.open_here()
print(f"Project loaded: {project.name} ({project.project_id})")

Project loaded: awr2944_workspace (3a182508)


## 1. Discover Hardware

First, we will scan the system to see what COM ports and network adapters are currently visible, regardless of configuration.

In [2]:
discovery = project.hardware.discover()

print("=== Serial Ports (pyserial) ===")
for p in discovery.serial_ports:
    print(f"{p.port:8s} {p.name} (XDS110: {p.is_xds110})")

print("\n=== COM Port Roles (heuristics) ===")
for p in discovery.com_ports:
    print(f"{p.com:8s} -> {p.likely_role} (conf={p.confidence})")

print("\n=== Network Adapters ===")
for a in discovery.network_adapters:
    print(f"{a.get('InterfaceAlias', '')}: {a.get('IPAddress', '')}")

=== Serial Ports (pyserial) ===
COM1     Communications Port (COM1) (XDS110: False)
COM3     AR-DevPack-EVM-012 (COM3) (XDS110: False)
COM4     AR-DevPack-EVM-012 (COM4) (XDS110: False)
COM5     AR-DevPack-EVM-012 (COM5) (XDS110: False)
COM6     AR-DevPack-EVM-012 (COM6) (XDS110: False)
COM7     XDS110 Class Auxiliary Data Port (COM7) (XDS110: True)
COM8     XDS110 Class Application/User UART (COM8) (XDS110: True)

=== COM Port Roles (heuristics) ===
COM7     -> awr_xds_uart_candidate (conf=high)
COM8     -> awr_rs232_candidate (conf=high)

=== Network Adapters ===
Loopback Pseudo-Interface 1: ::1
Ethernet 5: 192.168.33.30
Ethernet: 128.101.169.148
Loopback Pseudo-Interface 1: 127.0.0.1


## 2. Run Hardware Doctor

Now we run the full diagnostic suite. This will check file structures, executables, network bindings, and attempt read-only probes of the UART and DCA APIs.

In [3]:
report = project.doctor(include_hardware=True)
report.print()

Hardware Doctor Report (Mode: LIVE_DIAGNOSTIC)

Timestamp: 2026-07-15T00:38:15.028962+00:00

[FAIL] * project_structure              Missing: profiles, .awr2944

[FAIL] * portable_config_valid          Error parsing awr2944.toml: [Errno 2] No such file or directory: 
'C:\\Users\\khams008\\Documents\\awr2944-fmcw-radar\\awr2944.toml'

[FAIL] * local_config_valid             Error parsing local.toml: [Errno 2] No such file or directory: 
'C:\\Users\\khams008\\Documents\\awr2944-fmcw-radar\\.awr2944\\local.toml'

...

[SKIP] * dca_control_responds           Prerequisites failed

Depends on: dca_control_exe_exists, cf_json_exists, cf_json_consistency, host_nic_owns_ip

Summary:

✘ Found 8 errors.

⚠ System is NOT fully ready for capture.

## 3. Strict Verification

Uncomment the following line to ensure that an exception is raised if the system is not completely ready for capture.

In [4]:
# report.raise_for_errors(strict=True)